## Introduction

For day 10, the theme of the day is "air" and I tried to keep it simple making a map of "air temperature" using xarray.

In this code, a plot using the air temperature is done and this for Europe. The data are obtained from the [Climate Data Store](https://cds.climate.copernicus.eu/). In order to be able to download the data, please sign up for an account.

In what follows, we follow this [tutorial](https://ecmwf-projects.github.io/copernicus-training-c3s/reanalysis-climatology.html) here.

The code was run in Google Colab.




In [ ]:
!pip install cdsapi cartopy --quiet

## Importing the Python libraries

In [ ]:
# code obtained from the tutorial:
# https://ecmwf-projects.github.io/copernicus-training-c3s/reanalysis-climatology.html
import cdsapi

# Libraries for working with multidimensional arrays
import numpy as np
import requests
import xarray as xr

# Libraries for plotting and visualising data
import matplotlib.path as mpath
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.feature as cfeature

# Disable warnings for data download via API
import urllib3
urllib3.disable_warnings()

In my opinion, the easiest to pull the data is to use the following two lines with your API pasted. You can find the API key under your "profile" under the https://cds.climate.copernicus.eu/ website.

In [ ]:
import os
# Replace with your own CDS key and URL
cds_api_key = "FILL IN YOUR API KEY HERE"
cds_url = "https://cds.climate.copernicus.eu/api"

# Write config file - This comes in the root directory in Google Colab
os.makedirs(os.path.expanduser("~"), exist_ok=True)
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(f"url: {cds_url}\nkey: {cds_api_key}\n")

To see if the API key is installed successful:

In [ ]:
c = cdsapi.Client()
print("CDS API client successfully initialized!")

In order to look for the data you need, you can visit the datasets where you can select your choice. It will automatically generate the following code.

For the data for "air temperature", the dataset [ERA5 monthly averaged data on single levels from 1940 to present](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview) is used. In this form, you select Monthly averaged reanalysis/2m temperature/2024/select all (months)/Sub-region extraction/NetCDF4/unarchived.

For "Sub-region extraction" you can fill in the following numbers (but you can change this or select "Whole available region" in case you want the data for the entire globe.

In [ ]:
url = "https://raw.githubusercontent.com/EllenB/30-day-map-challenge-2025/main/day_10/cds_api_subregion.PNG"
r = requests.get(url)
with open("cds_api_subregion.PNG", "wb") as f:
    f.write(r.content)

from IPython.display import Image
Image("cds_api_subregion.PNG")

Subsequently, you expand the "Corresponding API request" and copy and paste the code below here:

In [ ]:
dataset = "reanalysis-era5-single-levels-monthly-means"
request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": ["2m_temperature"],
    "year": ["2024"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [72, -25, 34, 40]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()


Alternatively, you could have also created a directory and added a few lines like in the tutorial where you can avoid the following steps here below but I just copied and pasted the code generated from the CDS api and did the following steps.

You will see (if you work in Google Colab), a file in the "Files" section uploaded but it has a long name existing of random numbers and letters and with the .nc extension.

Right click on this file and then click on "Copy path":

In [ ]:
url2 = 'https://raw.githubusercontent.com/EllenB/30-day-map-challenge-2025/main/day_10/colab_files_png_3.png'


r = requests.get(url2)
with open("colab_files_png_3.png", "wb") as f:
    f.write(r.content)

from IPython.display import Image
Image("colab_files_png_3.png")

After which you can copy the path below with the file name you got:

In [ ]:
ds = xr.open_dataset('/content/ae7881e74ae26bdfbe024071bff9807d.nc')

In [ ]:
ds

Before procedding, we need to convert the longitude values as you can see in the output above, these are xx:

In [ ]:
ds.longitude

In [ ]:
ds_180 = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180)).sortby('longitude')

In [ ]:
# Create Xarray Data Array
da = ds_180['t2m']

In [ ]:
da

Convert everything in degrees:

In [ ]:
da_degc = da - 273.15

In [ ]:
da_degc = da_degc.assign_attrs(da.attrs)
da_degc.attrs['units'] = '° C'

Create a simple plot without any formatting and this only for the first month of the year 2024:

In [ ]:
da_degc[0,:,:].plot()

Change the figure a bit:

In [ ]:
fig = plt.figure(figsize=(14, 8), facecolor = '#FCD7FB')
ax = plt.subplot(1,1,1, projection=ccrs.PlateCarree())
ax.coastlines(color='white') # Add coastlines
im = plt.pcolormesh(da_degc.longitude, da_degc.latitude, da_degc[1,:,:], cmap='cool')

plt.title("Air temperature in Europe (January 2024)", size = 18)

cbar = plt.colorbar(im,fraction=0.046, pad=0.04, location = 'bottom', shrink = 0.3)
cbar.set_label(
    f"Air temperature (°C)",
    size=12,
    color='#303030'
)
txt = ax.text(0.02, 0.02, "Ellen Brock \ncds.climate.copernicus.eu \nECMWF",
              size=8,
              color='gray',
              transform = ax.transAxes)

plt.savefig(
    "air_temperature_europe_jan2024.png",
    dpi=300,
    bbox_inches='tight',
    facecolor=fig.get_facecolor()
)

plt.show()

## References

https://ecmwf-projects.github.io/copernicus-training-c3s/reanalysis-climatology.html